In [1]:
import pandas as pd
import os
import yaml
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
parameters_csv = pd.read_csv('/home/fsoto/Documents/LCsSSL/wandb_csv/vicreg_parameters.csv')


In [3]:
def find_config(pre_trained_lc_path, pre_trained_iteration):

    config_file = os.path.join(pre_trained_lc_path,str(pre_trained_iteration),'.hydra','config.yaml')
    with open(config_file, 'r') as f:
        config = yaml.safe_load(f)
    inv_weight = config['model']['inv_loss_weight']
    cov_weight = config['model']['cov_loss_weight']
    std_weight = config['model']['std_loss_weight']
    return inv_weight, cov_weight, std_weight


In [4]:
parameters_csv[['inv_weight', 'cov_weight', 'std_weight']] = parameters_csv.apply(
    lambda row: pd.Series(find_config(
        pre_trained_lc_path=row['model.pre_trained_lc_path'],
        pre_trained_iteration=row['model.pre_trained_iteration']
    )), axis=1
)

FileNotFoundError: [Errno 2] No such file or directory: '/home/fsoto/Documents/LCsSSL/logs/train/multiruns/VICReg_Parameters_Ablation_5_25/1/.hydra/config.yaml'

In [ ]:
#group rows by inv_weight, cov_weight, std_weight and calculate mean and std of val/f1 column 
grouped = parameters_csv.groupby(['inv_weight', 'cov_weight', 'std_weight']).agg(
    mean_val_f1=('val/f1', 'mean'),
    std_val_f1=('val/f1', 'std')
).reset_index()

In [ ]:
grouped

,inv_weight,cov_weight,std_weight,mean_val_f1,std_val_f1
0,1.0,1.0,1.0,0.365957,0.160546
1,5.0,1.0,5.0,0.679592,0.008096
2,5.0,1.0,25.0,0.687460,0.005368
3,10.0,1.0,10.0,0.674114,0.006924
4,25.0,1.0,5.0,0.603737,0.019908
5,25.0,1.0,25.0,0.666519,0.007496


In [ ]:
# Save grouped results to CSV
output_path = '/home/fsoto/Documents/LCsSSL/wandb_csv/vicreg_parameters_ablation_results.csv'
grouped.to_csv(output_path, index=False, float_format='%.6f')

print(f"✓ Results saved to: {output_path}")
print(f"  Rows: {len(grouped)}")
print(f"  Columns: {', '.join(grouped.columns)}")
print(f"\nPreview:")
print(grouped.to_string(index=False))
